### Système de stockage Delta Lake

In [35]:
from pathlib import Path
import pandas as pd
from deltalake import write_deltalake
from deltalake import DeltaTable

Attetion, à chaque fois que l'on relance ces blocs de code, certes les df sont re-initialisés, mais les fichiers textes sur lesquels ils sont écrits sont les mêmes. Par conséquent cela incrémente le nombre de versions.

In [36]:
df = pd.DataFrame({
     "id": [1, 2, 3],
     "name": ["Alice", "Bob", "Charlie"]
})
#base_path = Path(__file__).parent
base_path = Path.cwd() 
delta_path = base_path / "delta_demo"
write_deltalake(
      str(delta_path),
      df,
      mode="overwrite"
)
print(f"Table Delta créée dans : {delta_path.resolve()}")

Table Delta créée dans : /home/axel/Projet9/projet9/delta_demo


In [37]:
# Lecture simple
table = DeltaTable(str(delta_path))
print(table.to_pandas())
# Ajout de nouvelles données
df2 = pd.DataFrame({"id": [4], "name": ["Diana"]})
write_deltalake(
      str(delta_path),
      df2,
      mode="append"
)
print(f"Table Delta updated.")
# Relecture de la table pour vérifier l'insertion
table = DeltaTable(str(delta_path))
print(table.to_pandas())
# Vérifiez la nouvelle version
print("Version :", table.version())

   id     name
0   1    Alice
1   2      Bob
2   3  Charlie
Table Delta updated.
   id     name
0   4    Diana
1   1    Alice
2   2      Bob
3   3  Charlie
Version : 9


In [38]:
# Lecture simple
old_table = DeltaTable(str(delta_path), version=0)
print(old_table.to_pandas())

   id     name
0   1    Alice
1   2      Bob
2   3  Charlie


In [42]:
overwrite_df = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Axel", "Loïc", "Alexis"]
})
write_deltalake(str(delta_path), overwrite_df, mode="overwrite")
print("Table Delta remplacée par une nouvelle version.")

Table Delta remplacée par une nouvelle version.


In [43]:
# Lecture simple
old_table = DeltaTable(str(delta_path), version=0)
print(old_table.to_pandas())
# Lecture simple
old_table = DeltaTable(str(delta_path), version=1)
print(old_table.to_pandas())
# Lecture simple
old_table = DeltaTable(str(delta_path), version=2)
print(old_table.to_pandas())
# Lecture simple
old_table = DeltaTable(str(delta_path), version=3)
print(old_table.to_pandas())
# Lecture simple
old_table = DeltaTable(str(delta_path), version=4)
print(old_table.to_pandas())

   id     name
0   1    Alice
1   2      Bob
2   3  Charlie
   id     name
0   4    Diana
1   1    Alice
2   2      Bob
3   3  Charlie
   id     name
0   1    Alice
1   2      Bob
2   3  Charlie
   id     name
0   4    Diana
1   1    Alice
2   2      Bob
3   3  Charlie
   id    name
0   1    Axel
1   2    Loïc
2   3  Alexis


In [44]:
delta_path_partitioned = base_path / "delta_demo_partitioned"

In [45]:
write_deltalake(str(delta_path_partitioned), overwrite_df, partition_by=["name"])
print("📁 Table Delta partitionnée créée.")

DeltaError: Generic error: A table already exists at: file:///home/axel/Projet9/projet9/delta_demo_partitioned/

In [46]:
old_table = DeltaTable(str(delta_path_partitioned), version=0)
print(old_table.to_pandas())

   id    name
0   3  Alexis
1   1    Axel
2   2    Loïc


In [47]:
table = DeltaTable(str(delta_path))
table.delete("name = 'Alexis'")
print("Données supprimées pour Alexis.")

Données supprimées pour Alexis.


In [48]:
# Lecture simple
old_table = DeltaTable(str(delta_path), version=0)
print(old_table.to_pandas())
# Lecture simple
old_table = DeltaTable(str(delta_path), version=1)
print(old_table.to_pandas())

   id     name
0   1    Alice
1   2      Bob
2   3  Charlie
   id     name
0   4    Diana
1   1    Alice
2   2      Bob
3   3  Charlie


In [49]:
table = DeltaTable(str(delta_path))
print(table.version())           # version courante
print(table.history())   

12
[{'timestamp': 1782222876562, 'operation': 'DELETE', 'operationParameters': {'predicate': "name = 'Alexis'"}, 'readVersion': 11, 'engineInfo': 'delta-rs:py-1.6.0', 'operationMetrics': {'execution_time_ms': 71, 'num_added_files': 1, 'num_copied_rows': 2, 'num_deleted_rows': 1, 'num_removed_files': 1, 'rewrite_time_ms': 0, 'scan_time_ms': 60}, 'clientVersion': 'delta-rs.py-1.6.0', 'version': 12}, {'timestamp': 1782222865058, 'operation': 'WRITE', 'operationParameters': {'mode': 'Overwrite'}, 'engineInfo': 'delta-rs:py-1.6.0', 'clientVersion': 'delta-rs.py-1.6.0', 'operationMetrics': {'execution_time_ms': 2, 'num_added_files': 1, 'num_added_rows': 3, 'num_partitions': 0, 'num_removed_files': 1}, 'version': 11}, {'timestamp': 1782222800040, 'operation': 'WRITE', 'operationParameters': {'mode': 'Overwrite'}, 'engineInfo': 'delta-rs:py-1.6.0', 'clientVersion': 'delta-rs.py-1.6.0', 'operationMetrics': {'execution_time_ms': 1, 'num_added_files': 1, 'num_added_rows': 3, 'num_partitions': 0, 

In [50]:
from datetime import datetime, timezone
history = table.history()

rows = []
for h in history:
    rows.append({
        "version": h.get("version"),
        "operation": h.get("operation"),
        "timestamp_utc": datetime.fromtimestamp(
            h.get("timestamp", 0) / 1000, tz=timezone.utc
        ).isoformat(),
        "readVersion": h.get("readVersion"),
        "mode": (h.get("operationParameters") or {}).get("mode"),
        "predicate": (h.get("operationParameters") or {}).get("predicate"),
        "mergePredicate": (h.get("operationParameters") or {}).get("mergePredicate"),
        "num_added_rows": (h.get("operationMetrics") or {}).get("num_added_rows"),
        "num_output_rows": (h.get("operationMetrics") or {}).get("num_output_rows"),
        "num_updated_rows": (h.get("operationMetrics") or {}).get("num_updated_rows"),
        "num_deleted_rows": (h.get("operationMetrics") or {}).get("num_deleted_rows"),
        "execution_time_ms": (h.get("operationMetrics") or {}).get("execution_time_ms"),
        "engineInfo": h.get("engineInfo"),
    })

df_hist = pd.DataFrame(rows).sort_values("version")
print(df_hist.to_string(index=False))

 version operation                    timestamp_utc  readVersion      mode       predicate mergePredicate  num_added_rows num_output_rows num_updated_rows  num_deleted_rows  execution_time_ms        engineInfo
       0     WRITE 2026-06-22T22:46:13.165000+00:00          NaN Overwrite             NaN           None             3.0            None             None               NaN                  6 delta-rs:py-1.6.0
       1     WRITE 2026-06-22T22:47:43.875000+00:00          NaN    Append             NaN           None             1.0            None             None               NaN                  8 delta-rs:py-1.6.0
       2     WRITE 2026-06-23T13:13:08.382000+00:00          NaN Overwrite             NaN           None             3.0            None             None               NaN                160 delta-rs:py-1.6.0
       3     WRITE 2026-06-23T13:13:10.675000+00:00          NaN    Append             NaN           None             1.0            None             None      

In [58]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder.appName("delta-demo") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [60]:
delta_path = Path.cwd() / "delta_demo"
df = spark.read.format("delta").load(str(delta_path))
df.show()

26/06/23 18:03:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---+----+
| id|name|
+---+----+
|  1|Axel|
|  2|Loïc|
+---+----+



In [62]:
from delta.tables import DeltaTable
delta_table = DeltaTable.forPath(spark, str(delta_path))
delta_table.optimize().executeCompaction()

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [63]:
delta_table.optimize().executeCompaction().show(truncate=False)

+------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|path                                      |metrics                                                                                                                                   |
+------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|file:/home/axel/Projet9/projet9/delta_demo|{0, 0, {NULL, NULL, 0.0, 0, 0}, {NULL, NULL, 0.0, 0, 0}, 0, NULL, NULL, 0, 1, 1, 1, false, 0, 0, 1782240756995, 0, 8, 0, NULL, NULL, 2, 2}|
+------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------+



### Système de stockage Apache Iceberg

In [3]:
from pathlib import Path
import pyarrow as pa
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, LongType, StringType
from pyiceberg.catalog import load_catalog
from pyiceberg.exceptions import TableAlreadyExistsError
# --- Chemins locaux (à côté du script) ---
#base_path = Path(__file__).parent
base_path = Path.cwd()
warehouse_path = base_path / "iceberg_demo"
warehouse_path.mkdir(parents=True, exist_ok=True)
# --- Schéma Iceberg (champs optionnels pour matcher PyArrow par défaut) ---
schema = Schema(
                NestedField(1, "id", LongType(), required=False),
                NestedField(2, "name", StringType(), required=False),
)
# --- Catalogue local (SQLite) + warehouse local ---
catalog = load_catalog(
    "local",
    **{
        "type": "sql",
        "uri": f"sqlite:///{(base_path / 'iceberg_catalog.db').resolve()}",
        "warehouse": warehouse_path.resolve().as_uri(),
    }
)
# Namespace (création si besoin)
try:
    catalog.create_namespace("default")
except Exception:
    pass
identifier = "default.customers"
# Table : create-or-load
try:
            table = catalog.create_table(identifier, schema=schema)
            print("Table créée :", identifier)
except TableAlreadyExistsError:
            table = catalog.load_table(identifier)
            print("Table déjà existante, chargée :", identifier)
print("Location :", table.location())
# --- Écriture via PyArrow ---
arrow_table = pa.table(
                        {
                            "id": [1, 2, 3], "name": ["Alice", "Bob", "Charlie"],
                        }
)
table.append(arrow_table)
print("Données ajoutées (nouveau snapshot créé)")
# --- Lecture ---
print("\nContenu de la table :")
for row in table.scan().to_arrow().to_pylist():
                print(row)
# --- Snapshots ---
print("\nSnapshots disponibles :")
for snapshot in table.snapshots():
            print(snapshot.snapshot_id, snapshot.timestamp_ms)

Table créée : default.customers
Location : file:///home/axel/Projet9/projet9/iceberg_demo/default/customers
Données ajoutées (nouveau snapshot créé)

Contenu de la table :
{'id': 1, 'name': 'Alice'}
{'id': 2, 'name': 'Bob'}
{'id': 3, 'name': 'Charlie'}

Snapshots disponibles :
7634087672091833631 1782209670590
